### Compensation Loop Design eqs

---

Estas relaciones describen la ganancia DC del lazo de control de un flyback en modo corriente (CCM). No representan la dinámica completa, sino el comportamiento estático del sistema.

#### Variables clave

- D: define cuánta energía se transfiere por ciclo (duty cycle)
- M: relación entre la tensión reflejada del secundario y la entrada
- tau_L: representa la dinámica energética del convertidor (dependiente de Lp, carga y frecuencia)
- G0: ganancia DC del sistema de control

---

#### Interpretación del sistema

El convertidor flyback en modo corriente se comporta aproximadamente como:

- Un sistema de segundo orden (dos polos dominantes)
- Puede incluir un cero adicional debido al ESR del capacitor de salida

Modelo simplificado:

G(s) ≈ G0 / (1 + s/w_p1)(1 + s/w_p2)

donde:

- w_p1: polo dominante asociado al capacitor de salida y la carga
- w_p2: polo asociado a la dinámica del inductor (energía almacenada)
- ESR introduce un cero que puede mejorar la estabilidad

---

#### Significado físico

- D controla la energía transferida por ciclo
- M indica el acoplamiento entre primario y secundario
- tau_L define qué tan rápido responde el sistema a cambios de carga
- G0 determina qué tan sensible es la salida frente a cambios en el control

---

#### Uso en diseño

Estas ecuaciones se utilizan como punto de partida para:

- Diseñar la compensación del lazo (TL431 + optoacoplador o COMP del controlador)
- Estimar la ganancia inicial del sistema antes de compensar
- Ubicar polos y ceros de la red de compensación

---

#### Implementación en control

El sistema completo se compone de:

1. Planta (power stage)
   - Flyback (modelado por G0 y sus polos)

2. Sensor de corriente
   - Rcs (define límite de corriente)

3. Amplificador de error
   - TL431 o amplificador interno

4. Red de compensación
   - Introduce:
     - un cero dominante (para cancelar polo principal)
     - un polo de alta frecuencia (para ruido)

---

#### Objetivo de la compensación

- Convertir el sistema en uno dominado por un solo polo efectivo
- Mejorar margen de fase (>45° típico)
- Evitar oscilaciones y sobreimpulsos

---

#### Consideraciones importantes

- Este modelo es válido principalmente en CCM
- En DCM:
  - el sistema se comporta más cercano a primer orden
  - la compensación es más simple
- La ganancia DC (G0) sigue siendo útil como referencia inicial




In [7]:
# Import necessary libraries
import math
import numpy as np
from matplotlib.ticker import EngFormatter

# General parameters 

eta = 0.85    # Efficiency of the converter
P_OUT = 110  # Output power in watts
P_IN = P_OUT/eta  # Input power in watts
print(f"Input Power (P_IN): {EngFormatter(unit='W').format_data(P_IN)}")    
V_IN_min = 110  # Minimum input voltage in volts
V_IN_max = 132  # Maximum input voltage in volts
Vout = 45  # Output voltage in volts
f_sw = 110e3  # Switching frequency in hertz
RL = 18 # Load resistance in ohms
V_IN_rms_typical = 120  # Typical RMS input voltage in volts
Vref = 5 # Reference voltage from the controller in volts

V_bulk_typical = (np.sqrt(2)*V_IN_rms_typical-2*0.7)  # Estimated bulk voltage in volts
print(f"Estimated Typical Bulk Voltage (V_bulk_typical): {EngFormatter(unit='V').format_data(V_bulk_typical)}")

Rcs = 0.656 # Current sense resistor in ohms. Actualizar si algo

V_BULK_min = (np.sqrt(2)*V_IN_min-2*0.7)*0.9  # Minimum bulk voltage in volts, considering a 10% margin




Input Power (P_IN): 129.412 W
Estimated Typical Bulk Voltage (V_bulk_typical): 168.306 V


In [8]:
# Duty cycle (Ecuación 20)
def calc_D(Nps, Vout, Vbulk_min):
    return (Nps * Vout) / (Vbulk_min + (Nps * Vout))


# Parámetro M (Ecuación 22)
def calc_M(Vout, Nps, Vbulk_min):
    return (Vout * Nps) / Vbulk_min


# Tau_L (Ecuación 21)
def calc_tau_L(Lp, fsw, Rout, Nps):
    return (2 * Lp * fsw) / (Rout * (Nps**2))


# Ganancia DC (Ecuación 19)
def calc_G0(Rout, Nps, Rcs, Acs, D, tau_L, M):
    denominator = ((1 - D)**2) / tau_L + (2 * M) + 1
    return (Rout * Nps) / (Rcs * Acs) * (1 / denominator)

def calc_vbulk_max(V_IN_max):
    return np.sqrt(2) * V_IN_max

def calc_vreflected(V_DS_rated, V_BULK_max):
    return 0.8 * (V_DS_rated - 1.3 * V_BULK_max)

def calc_nps(V_REFLECTED, V_OUT):
    return V_REFLECTED / V_OUT

def calc_npa(N_PS, V_OUT, V_BIAS):
    return N_PS * (V_OUT / V_BIAS)

def calc_vdiode(V_BULK_max, N_PS, V_OUT):
    return (V_BULK_max / N_PS) + V_OUT

def calc_dmax(N_PS, V_OUT, V_F, V_BULK_min):
    return (N_PS * (V_OUT + V_F)) / (V_BULK_min + N_PS * (V_OUT + V_F))

def calc_lp(V_BULK_min, N_PS, V_OUT, P_IN, f_SW):
    return (0.5 * (V_BULK_min**2) * ((N_PS * V_OUT) / (V_BULK_min + N_PS * V_OUT))**2) / (0.1 * P_IN * f_SW)


In [9]:

Vbulk_max = calc_vbulk_max(V_IN_max)
V_DS_rated = 600  # Rated drain-source voltage of the MOSFET in volts
V_reflected = calc_vreflected(V_DS_rated, Vbulk_max)

Nps = calc_nps(V_reflected, Vout)  
lp = calc_lp(V_BULK_min, Nps, Vout, P_IN, f_sw)


Acs = 3 # Gain of the current sense amplifier datasheet.

M = calc_M(Vout, Nps, V_BULK_min)
print(f"Parameter M: {EngFormatter(unit='').format_data(M)}")

tau_L= calc_tau_L(lp, f_sw, RL, Nps)
print(f"Tau_L: {EngFormatter(unit='s').format_data(tau_L)}")

D = calc_D(Nps, Vout, V_BULK_min)
print(f"Duty Cycle (D): {EngFormatter(unit='').format_data(D)}")

G0 = calc_G0(RL, Nps, Rcs, Acs, D, tau_L, M)
print(f"DC Gain (G0): {EngFormatter(unit='').format_data(G0)}")
print(f"DC Gain (G0) in dB: {20 * np.log10(G0):.2f} dB")

#Rout → Vout / Iout
#Nps  → N_primary / N_secondary
#Vbulk_min → voltaje DC mínimo después del rectificador
#Rcs  → resistencia de sensado
#Acs  → ganancia del amplificador de corriente (≈1 si no hay amplificador externo)
#Lp   → inductancia de magnetización
#fsw  → frecuencia de switching



Parameter M: 2.06027
Tau_L: 928.237 ms
Duty Cycle (D): 673.232 m
DC Gain (G0): 11.0973
DC Gain (G0) in dB: 20.90 dB
